# DEMETER LSTM-AE — Rolling-Window Training

For each rolling window (12 months train, 3 months validation, 3-month stride):

1. Load the background-corrected window data and split it into train / validation.
2. Build half-orbit pair datasets: **all pairs** and **normal pairs** (seismic label 0).
3. Fit one scaler on the normal training pairs and apply it to all four datasets.
4. Train the autoencoder on normal train pairs, with early stopping on normal validation pairs.
   With `WARM_START`, window *i > 0* starts from the model saved for window *i − 1*.
5. Evaluate on **all** train and validation pairs: aggregate reconstruction error, threshold at the
   `pc`-th percentile of the training errors, and seismic association of the anomalies.

## Setup (PC or Google Colab)

- **PC:** set `LOCAL_PROJECT` to the repository folder.
- **Colab:** set `DRIVE_PROJECT` to the repository folder in your Google Drive. If it does not exist yet,
  it is cloned there from GitHub (code + data). An uploaded copy of the folder works too.

In [ ]:
import sys
import subprocess
from pathlib import Path

LOCAL_PROJECT = Path(r"D:\GIT\LSTM-AE-For-Demeter-Seismic-Study")
DRIVE_PROJECT = Path("/content/drive/MyDrive/LSTM-AE-For-Demeter-Seismic-Study")
REPO_URL = "https://github.com/Meghababu99/LSTM-AE-For-Demeter-Seismic-Study.git"
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)], check=True)
else:
    PROJECT = LOCAL_PROJECT

CODE_DIR = PROJECT / "Clean-Code"
sys.path.insert(0, str(CODE_DIR))
print("Project:", PROJECT)

In [ ]:
import random

import numpy as np
import pandas as pd
import torch

from lstm import (
    LSTMAutoencoder,
    SeismicCriteria,
    generate_windows,
    load_eq_catalog,
    prepare_window,
    train_window,
    evaluate_window,
)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Settings

In [ ]:
SEED = 201894
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Data
DATA_DIR = PROJECT / "Data"
BG_WINDOW_DIR = DATA_DIR / "Bg_window_data"
LABEL_DIR = DATA_DIR / "Label_data"
EQ_PATH = DATA_DIR / "EQ.csv"

# Rolling windows
train_months, val_months, stride = 12, 3, 3
WINDOW_START, WINDOW_END = "2005-01-01 00:00:00", "2010-01-02 00:00:00"
min_data_points = 17

# Anomaly threshold (percentile) and seismic criterion
pc = 98   # percentile
tw = 48   # time window (hours)
sw = 22   # spatial width (degrees)

# Model and training
input_size, hidden_size, num_layers, latent_dim = 11, 8, 2, 2
batch_size, lr, num_epochs, patience = 8, 0.001, 10, 150
RESTORE_BEST_EPOCH = False  # False: keep the model of the epoch where training stopped
COSINE_T_MAX = None         # None: constant learning rate

WARM_START = True           # window i > 0 starts from the model of window i-1 (False = "In-" re-initialisation)
RESUME = True               # skip windows already in the result CSV (continues an interrupted run)

m = f"tw{tw}-30dBG_{'' if WARM_START else 'In-'}A-re"  # run label

def build_model():
    return LSTMAutoencoder(input_size, hidden_size, num_layers, latent_dim)

OUTPUT_DIR_RESULTS = CODE_DIR / "outputs"
OUTPUT_MODEL_DIR = CODE_DIR / "Model-Retraining"

OUTPUT_DIR_RESULTS.mkdir(parents=True, exist_ok=True)
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
results_csv = OUTPUT_DIR_RESULTS / f"result_Hp_{m}.csv"
print("Run label:", m)
print("Outputs:", OUTPUT_DIR_RESULTS)

## Earthquake catalogue and rolling windows

In [ ]:
eq = load_eq_catalog(EQ_PATH)
seismic_criteria = SeismicCriteria(spatial_width=sw, time_window_hours=tw)
windows_train = generate_windows(WINDOW_START, WINDOW_END, train_months, val_months, stride)

print("Earthquake catalogue shape:", eq.shape)
print("Number of rolling windows:", len(windows_train))
for i, w in enumerate(windows_train):
    print(f"Window {i}: Train {w['train_start']} -> {w['train_end']} | Val {w['val_start']} -> {w['val_end']}")

## Training and evaluation loop

In [ ]:
done = pd.read_csv(results_csv) if RESUME and results_csv.exists() else pd.DataFrame(columns=["window"])
all_results = done.to_dict("records")

for i, w in enumerate(windows_train):
    if i in set(done["window"]):
        continue
    tag = f"Hp_{m}_w{i}"
    print(f"\n{'=' * 90}\nWindow {i}: Train {w['train_start']} -> {w['train_end']} | "
          f"Val {w['val_start']} -> {w['val_end']}\n{'=' * 90}")

    splits, datasets, scaled = prepare_window(
        BG_WINDOW_DIR / f"Background_data-window_{i}.pkl",
        {s: LABEL_DIR / f"summary_df_{s.lower()}_30D-{sw}SW-tw{tw}_w{i}.csv" for s in ("Train", "Val")},
        w,
        min_data_points,
    )

    prev_model = OUTPUT_MODEL_DIR / f"best_model_Hp_{m}_w{i-1}.pth" if WARM_START and i > 0 else None
    model = train_window(
        build_model(), scaled, tag, OUTPUT_MODEL_DIR, batch_size, prev_model,
        num_epochs=num_epochs, lr=lr, patience=patience,
        restore_best_epoch=RESTORE_BEST_EPOCH, cosine_t_max=COSINE_T_MAX,
    )

    all_results += evaluate_window(i, w, model, splits, datasets, scaled, eq, seismic_criteria, pc, m, OUTPUT_DIR_RESULTS)
    pd.DataFrame(all_results).to_csv(results_csv, index=False)  # saved after every window

## Results

In [ ]:
df_results = pd.DataFrame(all_results)
print(f"Results saved to {results_csv}")
display(df_results)